# Pseudo-labelling

Generates high-confidence labels automatically for the two easy classes, so hand-labelling
effort goes where judgement is actually needed — aquatic vegetation, wet mud and degraded
margin.

**What this buys, and what it does not.** Open water is genuinely rare at 10 m in these
waterholes: even at a permissive threshold the whole archive yields only a couple of
thousand pixels. What pseudo-labelling buys is *site coverage*. Hand labels put open water
at 2 sites; this puts it at 7. With grouped-by-site cross-validation the number of sites is
what limits generalisation, so that matters more than the pixel count.

**This is deliberately blind to vegetated water.** The open-water rule is an MNDWI
threshold, and sedges over standing water push MNDWI far negative. That is exactly the case
your hand labels exist to cover, so the rule stays conservative rather than being widened
until it "finds" the hard cases wrongly.

Masks are written to `labels_pseudo/`, separate from your hand labels, and tagged
`source: "pseudo"` so training can include or exclude them at will.

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "cookie-cutting" else Path.cwd() / "cookie-cutting"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import wh_config
import wh_inventory
import wh_footprint
import wh_plots
import wh_pseudo
import wh_tiles

cfg = wh_config.load()
manifest = wh_inventory.load_manifest(cfg)
print("config", cfg.source_path.name, "hash", cfg.hash)
print(f"{len(manifest):,} chips, {manifest['site_id'].nunique()} sites")

## Parameters

Calibrated against the data, not guessed. The originally configured values produced almost
nothing — see the comments for the sweeps that fixed them.

In [ ]:
PARAMS = wh_pseudo.PseudoParams(
    # Minimum clear scenes behind the monthly median. Was 4, which excluded
    # exactly the wet-season months where water exists (Jan/Feb average ~2
    # clear scenes) and yielded 30 pixels across 120 tiles.
    min_obs=2,

    # Open water. 0.15 is still far above typical land MNDWI (-0.4 to -0.6),
    # and a half-water mixed pixel sits near 0, so the threshold does the job
    # erosion used to.
    open_water_mndwi_min=0.15,
    open_water_ndvi_max=0.10,
    # Erosion OFF: 39% of water patches here are only 1-2 pixels, so one pass
    # deleted 35% of the water and two whole sites (7 sites -> 5).
    open_water_erode_px=0,

    # Surrounding vegetation: greener than most of the tile, well away from the
    # basin. The NDVI cut is a percentile of the tile's own distribution with a
    # floor, because dry-season savanna runs ~0.30-0.45 and any fixed threshold
    # high enough for the wet season yields nothing in the dry.
    vegetation_buffer_px=20,
    vegetation_ndvi_min=0.35,
    vegetation_ndvi_percentile=60.0,

    # Per class, per tile. Surrounding vegetation is most of a 1.5 km chip, so
    # uncapped it yields ~5,000 px/tile — enough to bury the hand labels the
    # model actually needs to learn from.
    max_pixels_per_class_per_tile=300,
)

# Which sites and months to generate for. Start with the sites you have hand
# labels for, so the two are comparable.
# SITES = ["002", "003", "004", "005", "006", "009", "013", "016", "025"]
SITES = None       # None = all sites
MONTHS = None      # None = every month

PARAMS

## Survey before writing

Counts what *would* be generated, without touching disk. Worth doing every time you change
a threshold — the yields here are small and sensitive, and it is much cheaper to find that
out now than after writing several hundred masks.

Counts are shown **after** the per-tile cap, so this is what would actually land.

In [ ]:
survey = wh_pseudo.survey(
    manifest, cfg, PARAMS, sites=SITES, months=MONTHS, max_tiles_per_site=20
)

print(f"surveyed {len(survey)} tiles\n")
for class_name in ("open_water", "surrounding_vegetation"):
    hit = survey[survey[class_name] > 0]
    print(f"{class_name:24s} {survey[class_name].sum():>8,} px  "
          f"{len(hit):>3d} tiles  {hit['site_id'].nunique()} sites")

skipped = survey[survey["vegetation_skipped"] != ""]
if len(skipped):
    print(f"\nvegetation skipped on {len(skipped)} tiles:")
    print(skipped["vegetation_skipped"].value_counts().to_string())
    print(f"affected sites: {sorted(skipped['site_id'].unique())}")

survey.groupby("site_id")[["open_water", "surrounding_vegetation"]].sum()

### Sites without a basin footprint

Surrounding vegetation is defined as "far outside the basin", which needs a footprint.
Sites without one are skipped rather than guessed at — putting basin pixels into the
majority class would teach the classifier the exact opposite of what we want.

If a site you care about is skipped, run `footprint_estimation.ipynb` for it first.

## Generate

Writes masks and sidecars to `labels_pseudo/`. Your hand labels in `labels/` are never
touched. Re-running overwrites the pseudo masks, which is safe.

In [ ]:
written = wh_pseudo.generate(manifest, cfg, PARAMS, sites=SITES, months=MONTHS)

print(f"wrote {len(written)} pseudo-label masks to {wh_pseudo.pseudo_label_dir(cfg)}")
if len(written):
    totals = written[["open_water", "surrounding_vegetation"]].sum()
    print(f"\n{totals['open_water']:,} open water px, "
          f"{totals['surrounding_vegetation']:,} surrounding vegetation px")
    print(f"\nsites covered: open water "
          f"{written.loc[written['open_water'] > 0, 'site_id'].nunique()}, "
          f"vegetation {written.loc[written['surrounding_vegetation'] > 0, 'site_id'].nunique()}")
written.head(10)

## What changed

Compare site coverage before and after. Site count is the number that matters — it is what
grouped cross-validation actually has to work with.

In [ ]:
import json as _json

def coverage(directory, source_name):
    rows = []
    for path in sorted(Path(directory).glob("*_labels.json")):
        meta = _json.loads(path.read_text())
        for class_name, count in meta["pixel_counts"].items():
            if count:
                rows.append({"class": class_name, "site_id": meta["site_id"],
                             "pixels": count, "source": source_name})
    return pd.DataFrame(rows)

hand = coverage(cfg.paths["labels"], "manual")
pseudo = coverage(wh_pseudo.pseudo_label_dir(cfg), "pseudo")
both = pd.concat([hand, pseudo], ignore_index=True)
both = both[both["class"] != "unlabelled"]

summary = both.pivot_table(
    index="class", columns="source", values="site_id",
    aggfunc=lambda s: s.nunique(), fill_value=0,
)
summary.columns = [f"{c}_sites" for c in summary.columns]
pixels = both.pivot_table(index="class", columns="source", values="pixels",
                          aggfunc="sum", fill_value=0)
pixels.columns = [f"{c}_px" for c in pixels.columns]
pd.concat([summary, pixels], axis=1)

## Do the pseudo-labels agree with you?

The visual check below is useful, but this is the decisive test: on pixels labelled by
**both** a person and the rule, do they say the same thing?

Read the rows. Each row is a class you painted by hand; the columns are what the automatic
rule called those same pixels. The diagonal is agreement.

In [ ]:
crosstab, summary = wh_pseudo.agreement_with_manual(cfg)
print(f"{summary['overlapping_pixels']:,} pixels labelled by both, "
      f"across {summary['overlapping_tiles']} tiles")
print(f"agreement: {summary.get('agreed', 0):,} "
      f"({100 * summary.get('agreement_rate', 0):.1f}%)\n")
crosstab

In [ ]:
figure = wh_plots.plot_pseudo_agreement(crosstab)
plt.show()

**How to read a bad row.** A rule that disagrees with you systematically — one row sending
most of its pixels to a single wrong column — is not noisy, it is wrong about something
specific. Chase that down before training on it, because a systematic error is exactly the
kind a classifier will learn faithfully.

Sporadic disagreement at class boundaries is expected and harmless: mixed pixels are the
reason the labelling guidance says to leave ambiguous margins unlabelled.

## Look at the masks

Plots the tiles where the rule actually claimed something, one per site so the sample spans
waterholes rather than showing six months of the same one.

Each row shows:

- **RGB + pseudo labels** — what was claimed;
- **MNDWI** with the open-water threshold drawn as a green contour;
- **NDVI** with the vegetation cutoff drawn in blue (a percentile of *this tile's* own
  distribution, so it moves between wet and dry season);
- **hand labels** for the same tile, where you have painted it.

The contours are the point: they show *why* a pixel was claimed, so a badly placed threshold
is visible as a contour in the wrong place rather than an unexplained blob. The cyan outline
is the basin footprint, which bounds where surrounding vegetation may be claimed.

In [ ]:
INSPECT_CLASS = "open_water"     # or "surrounding_vegetation"
N_TILES = 5

to_inspect = wh_pseudo.most_informative(cfg, n=N_TILES, class_name=INSPECT_CLASS)
print(f"tiles with the most {INSPECT_CLASS}, one per site:")
print(to_inspect[["site_id", "year_month", "open_water", "surrounding_vegetation"]]
      .to_string(index=False))

In [ ]:
for _, row in to_inspect.iterrows():
    tile = wh_tiles.read_tile(row["tif_path"], cfg)
    pseudo_mask = wh_tiles.read_mask(row["mask_path"], tile.shape)

    try:
        footprint = wh_footprint.load_mask(cfg, row["site_id"])
    except FileNotFoundError:
        footprint = None

    manual_path = cfg.paths["labels"] / Path(row["mask_path"]).name
    manual_mask = (
        wh_tiles.read_mask(manual_path, tile.shape) if manual_path.exists() else None
    )

    figure = wh_plots.plot_pseudo_labels(
        tile, pseudo_mask, cfg, PARAMS,
        footprint=footprint, manual_mask=manual_mask,
    )
    plt.show()

### Inspecting a specific tile

Set the site and month directly when you want to look at something the ranking above did not
surface — a tile you are suspicious of, or one where hand and pseudo labels disagreed.

In [ ]:
SITE = "002"
YEAR_MONTH = "2021-04"

written = wh_pseudo.written_masks(cfg)
match = written[(written["site_id"] == SITE) & (written["year_month"] == YEAR_MONTH)]

if match.empty:
    print(f"no pseudo mask for site {SITE} {YEAR_MONTH}")
else:
    row = match.iloc[0]
    tile = wh_tiles.read_tile(row["tif_path"], cfg)
    pseudo_mask = wh_tiles.read_mask(row["mask_path"], tile.shape)
    try:
        footprint = wh_footprint.load_mask(cfg, SITE)
    except FileNotFoundError:
        footprint = None
    manual_path = cfg.paths["labels"] / Path(row["mask_path"]).name
    manual_mask = (
        wh_tiles.read_mask(manual_path, tile.shape) if manual_path.exists() else None
    )
    figure = wh_plots.plot_pseudo_labels(
        tile, pseudo_mask, cfg, PARAMS, footprint=footprint, manual_mask=manual_mask
    )
    plt.show()